# Principio de Inversión de Dependencias (DIP) — Cafetería

**Dominio propio:** el servicio de ventas de la cafetería necesita consultar el inventario de insumos.

El DIP dice que un módulo de alto nivel no debe depender de una implementación concreta de bajo nivel, sino de una **abstracción**, y que esa implementación se **inyecta**.

Muestro primero el servicio atado a una clase concreta (viola) y luego el servicio dependiendo de una abstracción inyectada (cumple).

## Versión que VIOLA el DIP

`ServicioVentas` (alto nivel) instancia directamente `InventarioMemoria` (bajo nivel, concreto). Si mañana el inventario viene de una API o una base de datos, hay que **modificar** el servicio.

In [1]:
class InventarioMemoria:
    def __init__(self) -> None:
        self.stock: dict[str, int] = {"cafe": 500, "leche": 300}

    def disponible(self, insumo: str, cantidad: int) -> bool:
        return self.stock.get(insumo, 0) >= cantidad

    def descontar(self, insumo: str, cantidad: int) -> None:
        self.stock[insumo] = self.stock.get(insumo, 0) - cantidad


class ServicioVentas:
    def __init__(self) -> None:
        # Dependencia rigida: instancia la clase concreta (viola DIP)
        self.inventario = InventarioMemoria()
        self.vendidos: int = 0

    def vender(self, insumo: str, cantidad: int) -> str:
        if self.inventario.disponible(insumo, cantidad):
            self.inventario.descontar(insumo, cantidad)
            self.vendidos += 1
            return f"Vendido {cantidad} de {insumo}"
        return f"Sin stock de {insumo}"

In [2]:
servicio = ServicioVentas()
print(servicio.vender("cafe", 20))
print(servicio.vender("leche", 400))  # sin stock

Vendido 20 de cafe
Sin stock de leche


### Problema

`ServicioVentas` está soldado a `InventarioMemoria`. Para usar un inventario en base de datos o para probar el servicio con un doble de prueba, hay que editar la clase de alto nivel. Viola el DIP.

## Versión que CUMPLE el DIP

Defino la abstracción `FuenteInventario`. `ServicioVentas` depende de esa abstracción y la recibe **inyectada** por el constructor. Las implementaciones concretas se pueden intercambiar sin tocar el servicio.

In [3]:
from abc import ABC, abstractmethod

class FuenteInventario(ABC):
    @abstractmethod
    def disponible(self, insumo: str, cantidad: int) -> bool: ...

    @abstractmethod
    def descontar(self, insumo: str, cantidad: int) -> None: ...


class InventarioMemoria(FuenteInventario):
    def __init__(self) -> None:
        self.stock: dict[str, int] = {"cafe": 500, "leche": 300}
        self.movimientos: int = 0

    def disponible(self, insumo: str, cantidad: int) -> bool:
        return self.stock.get(insumo, 0) >= cantidad

    def descontar(self, insumo: str, cantidad: int) -> None:
        self.stock[insumo] = self.stock.get(insumo, 0) - cantidad
        self.movimientos += 1


class InventarioAPI(FuenteInventario):
    """Otra implementacion (simula una API remota)."""
    def __init__(self, url: str) -> None:
        self.url: str = url
        self.consultas: int = 0

    def disponible(self, insumo: str, cantidad: int) -> bool:
        self.consultas += 1
        print(f"[API {self.url}] consultando {insumo}")
        return True  # la API remota siempre confirma en este ejemplo

    def descontar(self, insumo: str, cantidad: int) -> None:
        print(f"[API {self.url}] descontando {cantidad} de {insumo}")


class ServicioVentas:
    """Alto nivel: depende de la abstraccion, no de la implementacion."""
    def __init__(self, inventario: FuenteInventario) -> None:
        self.inventario: FuenteInventario = inventario  # inyeccion de dependencia
        self.vendidos: int = 0

    def vender(self, insumo: str, cantidad: int) -> str:
        if self.inventario.disponible(insumo, cantidad):
            self.inventario.descontar(insumo, cantidad)
            self.vendidos += 1
            return f"Vendido {cantidad} de {insumo}"
        return f"Sin stock de {insumo}"

In [4]:
# La misma clase de alto nivel funciona con cualquier implementacion inyectada
servicio_local = ServicioVentas(InventarioMemoria())
print(servicio_local.vender("cafe", 20))

servicio_api = ServicioVentas(InventarioAPI("inventario.cafe/api"))
print(servicio_api.vender("leche", 400))

Vendido 20 de cafe
[API inventario.cafe/api] consultando leche
[API inventario.cafe/api] descontando 400 de leche
Vendido 400 de leche


### Análisis

`ServicioVentas` ya no conoce ninguna clase concreta: depende de `FuenteInventario`. Cambiar de `InventarioMemoria` a `InventarioAPI` (o a un doble de prueba) es cuestión de **inyectar** otra implementación en el constructor, sin tocar el servicio. Eso es inversión de dependencias real.